# Task 2 final audit: CPU export
Choose **Runtime → Change runtime type → CPU**, then run the cell. It reads already saved records and downloads one ZIP. Attach that ZIP in this chat. No training, inference or GPU is needed. Existing experiment files are left unchanged.


In [ ]:
# Run in a Colab CPU runtime. Reads saved evidence; performs no model inference.
from pathlib import Path, PurePosixPath
import hashlib, json, shutil, subprocess, zipfile
from datetime import datetime, timezone
from uuid import uuid4
from google.colab import drive, files

probe = subprocess.run(['sh', '-c', 'command -v nvidia-smi >/dev/null && nvidia-smi -L'], capture_output=True, text=True)
if 'GPU ' in probe.stdout:
    raise RuntimeError('Switch this notebook to a CPU runtime first. This export needs no GPU.')
drive.mount('/content/drive')
root = Path('/content/drive/MyDrive/ATML-Assignment-2/task2/experiment_v1')
assert json.loads((root/'progress.json').read_text())['status'] == 'GPU_WORK_COMPLETE', 'Expected the completed experiment.'
name = 'task2_audit_records_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid4().hex[:8]
out = Path('/content') / name
out.mkdir(exist_ok=False)

def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024*1024), b''): h.update(block)
    return h.hexdigest()

def write(relative, payload):
    p = out / relative
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_bytes(payload)

runs = ['standard', 'clip_005', 'central_8', 'clip_050', 'kl_000', 'kl_020']
jobs = [(root/'training'/n/'final.zip', 'summary.json') for n in runs]
for n in ['midpoint'] + runs:
    paths = sorted((root/'evaluation'/n).glob('prompt_*.zip'))
    assert len(paths) == 200, (n, len(paths))
    jobs += [(p, 'record.json') for p in paths]
paths = sorted((root/'cache_geometry').glob('row_*.zip'))
assert len(paths) == 32
jobs += [(p, 'record.json') for p in paths]
archive_provenance = []
for i,(path,required) in enumerate(jobs, 1):
    receipt = json.loads(path.with_suffix('.sha256.json').read_text())
    actual = digest(path)
    assert actual == receipt['sha256'] and receipt['archive'] == path.name, str(path)
    relative = path.relative_to(root).with_suffix('').as_posix()
    with zipfile.ZipFile(path) as z:
        manifest = json.loads(z.read('FILES_SHA256.json'))
        names = z.namelist()
        assert len(names) == len(set(names)) and set(names) == set(manifest) | {'FILES_SHA256.json'}
        assert required in manifest
        selected = []
        for member,expected in manifest.items():
            pure = PurePosixPath(member)
            assert not pure.is_absolute() and '..' not in pure.parts
            # Keep original JSON evidence/configuration only, never model weights.
            if pure.suffix != '.json': continue
            data = z.read(member)
            assert hashlib.sha256(data).hexdigest() == expected, (relative, member)
            write(relative + '/' + member, data)
            selected.append(member)
        write(relative + '/ORIGINAL_FILES_SHA256.json', z.read('FILES_SHA256.json'))
    archive_provenance.append({'archive':path.relative_to(root).as_posix(), 'receipt':receipt,
                               'verified_json_members':selected})
    if i % 100 == 0 or i == len(jobs): print(f'Verified saved records: {i}/{len(jobs)}', flush=True)
# Contracts, candidate freeze, session timing, compatibility record and generation settings.
# Existing CPU result archives and per-update tensor checkpoint archives are not copied.
for path in sorted(root.rglob('*.json')):
    relative = path.relative_to(root)
    if relative.parts[0] in ('cpu_results','audit_exports') or path.name.endswith('.sha256.json'): continue
    write('metadata/' + relative.as_posix(), path.read_bytes())
write('archive_provenance.json', (json.dumps(archive_provenance, indent=2)+'\n').encode())
write('EXPORT_SCOPE.json', (json.dumps({'source':str(root),'training_summaries':6,
    'evaluation_records':1400,'cache_records':32,'gpu_used':False,
    'original_results_unchanged':True,'model_weights_included':False,
    'verification':'Every source ZIP hash checked against its Drive receipt; each exported member checked against the original member manifest.'}, indent=2)+'\n').encode())
manifest = {p.relative_to(out).as_posix():digest(p) for p in sorted(out.rglob('*')) if p.is_file()}
write('FILES_SHA256.json',(json.dumps(manifest,indent=2)+'\n').encode())
archive = out.with_suffix('.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in sorted(out.rglob('*')):
        if p.is_file(): z.write(p,p.relative_to(out).as_posix())
saved = root/'audit_exports'/archive.name
saved.parent.mkdir(parents=True,exist_ok=True)
shutil.copyfile(archive,saved)
assert digest(saved) == digest(archive)
print('AUDIT EXPORT VERIFIED:', saved)
print('SHA256:',digest(archive))
print('No training or evaluation was rerun. Downloading the audit ZIP now.')
files.download(str(archive))
